# Practical 4: Basic Attack Classification Using Calibrated Heuristic Engine
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Semester:** 5th Semester B.E. / B.Tech  
**Topic:** Ground-Truth Attack Categorization, Multi-Pass Normalization, SIEM Severity Triage, and Heuristic Attribution  
**Pipeline Input:** Practical 3 Preprocessed Dataset (`preprocessed_logs.csv` — 300,010 records)  
**External Real-World Benchmark:** Elastic Official Production Web Log (`elastic_preprocessed_logs.csv` — 10,000 records)  

---

## 1. AIM & OBJECTIVES
### Aim
To assign high-precision, ground-truth multi-class attack labels and human-readable diagnostic attributions to preprocessed web access logs using a calibrated heuristic classification engine across 10 discrete attack vectors plus benign traffic.

### Objectives
1. **Pipeline Continuity:** Ingest the clean tabular dataset output from Practical 3 (`preprocessed_logs.csv`).
2. **Multi-Class Ground-Truth Labeling:** Map requests to 10 canonical attack classes (`ddos_http_flood`, `brute_force`, `sqli`, `xss`, `command_injection`, `path_traversal`, `dir_fuzzing`, `ssrf`, `web_shell`, `mal_scanner`) and `benign`.
3. **Zero False-Positive Protection:** Employ context-aware syntax grammar rules that prevent false positives on legitimate customer names and searches.
4. **SIEM Threat Triage:** Assign operational security severity ratings (`CRITICAL`, `HIGH`, `MEDIUM`, `LOW`, `INFO`) and generate human-readable `label_reason` attributions.
5. **Dual-Format Serialization:** Export the labeled dataset to CSV and high-efficiency Snappy-compressed Parquet.
6. **External Production Benchmark:** Validate heuristic precision against Elastic's official production web server logs.

---
## 2. THEORETICAL ARCHITECTURE: HEURISTIC LABELING VS ML DECOUPLING

### 2.1 The Critical Principle of Non-Circular Heuristics
A catastrophic flaw in naive machine learning projects is **circular data leakage**: copying regex signature matches as ML features (e.g., `feature_sqli = 1 if SQL_REGEX`). When this occurs, models memorize regex strings rather than learning underlying statistical and behavioral signals.

In our architecture:
- **Practical 4 (This Practical):** Applies rule-based signatures to establish **Ground-Truth Target Labels** (`label`).
- **Practical 5 (Feature Extraction):** Extracts **28 purely statistical and structural features** (`url_entropy`, `payload_length`, `special_char_ratio`, `ip_request_rate_1m`). **No heuristic regex is ever included in the feature set!**

### 2.2 Canonical Attack Classes & Priority Hierarchy

| Priority | Attack Class | Target Vector | Severity |
|---|---|---|---|
| 1 | `ddos_http_flood` | Volumetric session inter-arrival time ($\Delta t \le 0.04$s) | HIGH |
| 2 | `mal_scanner` | Automated audit tool signatures in User-Agent (`sqlmap`, `nikto`) | LOW |
| 3 | `command_injection` | Shellshock / OS command binaries and execution operators | CRITICAL |
| 4 | `sqli` | Database syntax grammar, union queries, tautologies (`'1'='1`) | HIGH |
| 5 | `xss` | Executable HTML/JS tags, DOM event handlers (`onerror=`) | HIGH |
| 6 | `path_traversal` | Directory climbing (`../`), sensitive OS file targets (`/etc/passwd`) | MEDIUM |
| 7 | `ssrf` | Cloud metadata service (`169.254.169.254`), private loopbacks | HIGH |
| 8 | `web_shell` | Backdoor filenames (`c99.php`, `wso.php`), execution parameters | CRITICAL |
| 9 | `dir_fuzzing` | Sensitive resource probes (`.env`, `.git`, `phpmyadmin`) | LOW |
| 10 | `brute_force` | Credential spraying against `/login`, `/auth` with 401/403 | MEDIUM |
| 11 | `benign` | Normal, legitimate browsing traffic | INFO |

---
## 3. IMPORT LIBRARIES & SETUP ENVIRONMENT
We import `pandas`, `numpy`, and our master heuristic classification engine.

In [ ]:
import sys
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Resolve directories
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORKSPACE_DIR = PRACTICAL_DIR.parent

sys.path.insert(0, str(WORKSPACE_DIR))
from src.pipeline.heuristic_engine import HeuristicLabeler

PROCESSED_P3 = WORKSPACE_DIR / 'Practical 3' / 'data' / 'processed'
PROCESSED_P4 = PRACTICAL_DIR / 'data' / 'processed'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'
REPORTS_DIR = PRACTICAL_DIR / 'outputs' / 'reports'

PROCESSED_P4.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_INPUT = PROCESSED_P3 / 'preprocessed_logs.csv'
EXTERNAL_INPUT = PROCESSED_P3 / 'elastic_preprocessed_logs.csv'

print(f"Practical 4 Environment Initialized!")
print(f"Ingesting from Practical 3: {PRIMARY_INPUT} (Exists: {PRIMARY_INPUT.exists()})")


---
## 4. INGEST PREPROCESSED DATASET & CALCULATE TEMPORAL VELOCITY
We load the clean tabular records from Practical 3 and calculate inter-request time deltas per client IP.

In [ ]:
t0 = time.time()
df = pd.read_csv(PRIMARY_INPUT)
print(f"Loaded Primary Dataset: {len(df):,} rows x {len(df.columns)} columns in {time.time()-t0:.2f}s")

# Compute inter-request time delta per client IP
df['_dt'] = pd.to_datetime(df['datetime_utc'], errors='coerce')
df['inter_req_time'] = df.groupby('client_ip')['_dt'].diff().dt.total_seconds()
df.drop(columns=['_dt'], inplace=True)

print(f"Calculated temporal arrival deltas. Sample head:")
df[['client_ip', 'datetime_utc', 'resource_requested', 'inter_req_time']].head(3)


---
## 5. BATCH HEURISTIC CLASSIFICATION PIPELINE
We apply the `HeuristicLabeler` across all records to generate ground truth labels, attribution reasons, and SIEM severity ratings.

In [ ]:
SEVERITY_MAP = {
    'command_injection': 'CRITICAL',
    'web_shell': 'CRITICAL',
    'sqli': 'HIGH',
    'xss': 'HIGH',
    'ssrf': 'HIGH',
    'ddos_http_flood': 'HIGH',
    'path_traversal': 'MEDIUM',
    'brute_force': 'MEDIUM',
    'dir_fuzzing': 'LOW',
    'mal_scanner': 'LOW',
    'benign': 'INFO'
}

labeler = HeuristicLabeler()
labels = []
reasons = []

t_class = time.time()
resources = df['resource_requested'].tolist()
methods = df['request_method'].astype(str).tolist()
statuses = df['status_code'].astype(int).tolist()
user_agents = df['user_agent'].astype(str).tolist()
irts = df['inter_req_time'].tolist()

for res, meth, stat, ua, irt in zip(resources, methods, statuses, user_agents, irts):
    irt_val = irt if (irt is not None and not np.isnan(irt)) else None
    lbl, rsn = labeler.classify_single_request(
        resource=res,
        method=meth,
        status_code=stat,
        user_agent=ua,
        inter_request_time=irt_val
    )
    labels.append(lbl)
    reasons.append(rsn)

df['label'] = pd.Series(labels, dtype='category')
df['label_reason'] = reasons
df['is_attack'] = (df['label'] != 'benign').astype(np.int8)
df['severity'] = pd.Series([SEVERITY_MAP.get(l, 'INFO') for l in labels], dtype='category')

elapsed = time.time() - t_class
print(f"Classification Completed in {elapsed:.2f}s ({len(df)/elapsed:,.0f} rows/sec)!")
print("\n--- Multi-Class Attack Distribution ---")
print(df['label'].value_counts())


---
## 6. EXTERNAL BENCHMARK: ELASTIC PRODUCTION TELEMETRY
We evaluate the heuristic classifier against Elastic's official production web access log (10,000 records) to verify false positive suppression on live browsing traffic.

In [ ]:
df_ext = pd.read_csv(EXTERNAL_INPUT)
df_ext['_dt'] = pd.to_datetime(df_ext['datetime_utc'], errors='coerce')
df_ext['inter_req_time'] = df_ext.groupby('client_ip')['_dt'].diff().dt.total_seconds()
df_ext.drop(columns=['_dt'], inplace=True)

ext_labels = []
ext_reasons = []
for r, m, s, u, irt in zip(df_ext['resource_requested'], df_ext['request_method'].astype(str), df_ext['status_code'].astype(int), df_ext['user_agent'].astype(str), df_ext['inter_req_time']):
    irt_val = irt if (irt is not None and not np.isnan(irt)) else None
    lbl, rsn = labeler.classify_single_request(r, m, s, u, irt_val)
    ext_labels.append(lbl)
    ext_reasons.append(rsn)

df_ext['label'] = pd.Series(ext_labels, dtype='category')
df_ext['label_reason'] = ext_reasons
df_ext['is_attack'] = (df_ext['label'] != 'benign').astype(np.int8)
df_ext['severity'] = pd.Series([SEVERITY_MAP.get(l, 'INFO') for l in ext_labels], dtype='category')

print("=" * 65)
print(f"External Elastic Production Evaluation ({len(df_ext):,} records):")
print(df_ext['label'].value_counts())
print("=" * 65)


---
## 7. EXPORT LABELED DATASETS (CSV & PARQUET)
We export the labeled ground-truth datasets for downstream Practical 5 feature extraction.

In [ ]:
csv_out = PROCESSED_P4 / 'labeled_logs.csv'
pq_out = PROCESSED_P4 / 'labeled_logs.parquet'
ext_out = PROCESSED_P4 / 'elastic_labeled_logs.csv'

df.drop(columns=['inter_req_time'], inplace=True, errors='ignore')
df_ext.drop(columns=['inter_req_time'], inplace=True, errors='ignore')

df.to_csv(csv_out, index=False)
df.to_parquet(pq_out, index=False, compression='snappy')
df_ext.to_csv(ext_out, index=False)

print(f"Exported Primary CSV     : {csv_out.name} ({csv_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported Primary Parquet : {pq_out.name} ({pq_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported External CSV    : {ext_out.name} ({ext_out.stat().st_size / (1024*1024):.2f} MB)")


---
## 8. PUBLICATION-QUALITY DIAGNOSTIC PLOTS
We visualize the attack class distribution, macro balance, and severity breakdown.

In [ ]:
# 1. Attack Class Distribution
plt.figure(figsize=(10, 4.5), dpi=150)
c_counts = df['label'].value_counts()
colors = ['#10B981' if c == 'benign' else '#EF4444' for c in c_counts.index]
plt.barh(c_counts.index.astype(str), c_counts.values, color=colors, edgecolor='#060912')
plt.title('Multi-Class Ground-Truth Attack Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Record Count', fontsize=10)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 2. Severity Breakdown
plt.figure(figsize=(7, 4), dpi=150)
s_counts = df['severity'].value_counts()
sev_colors = {'CRITICAL': '#991B1B', 'HIGH': '#EF4444', 'MEDIUM': '#F59E0B', 'LOW': '#3B82F6', 'INFO': '#10B981'}
plt.bar(s_counts.index.astype(str), s_counts.values, color=[sev_colors.get(s, '#94A3B8') for s in s_counts.index], width=0.55)
plt.title('Threat Severity Distribution (SOC Incident Response)', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=10)
plt.tight_layout()
plt.show()
